In [3]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/480.6 kB ? eta -:--:--
   - ------------------------------------- 20.5/480.6 kB 682.7 kB/s eta 0:00:01
   --- ----------------------------------- 41.0/480.6 kB 495.5 kB/s eta 0:00:01
   -------- ----------------------------- 102.4/480.6 kB 845.5 kB/s eta 0:00:01
   ------------------- -------------------- 235.5/480.6 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------  471.0/480.6 kB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 480.6/480.6 kB 2.2 MB/s eta 0:00:00


In [4]:
# ============================================================
# generate_prediksi.py
# Script untuk generate prediksi harga Rica 7 hari ke depan
# Output: CSV + MySQL
# ============================================================

import pandas as pd
import numpy as np
import joblib
import requests
import mysql.connector
from datetime import datetime, timedelta

# ── 1. Load model & scaler ──────────────────────────────────
model        = joblib.load('model_rf_rica_manado_final.pkl')
scaler       = joblib.load('scaler_rica_manado_final.pkl')
feature_cols = joblib.load('feature_cols.pkl')

print("Model loaded.")

# ── 2. Load histori harga dari CSV ──────────────────────────
df = pd.read_csv('clean_dataset_rica_manado_final.csv')
df['Tanggal'] = pd.to_datetime(df['Tanggal'])
df = df.sort_values('Tanggal').reset_index(drop=True)

df['Rolling_Mean_7']  = df['Harga'].rolling(window=7).mean()
df['Rolling_Mean_14'] = df['Harga'].rolling(window=14).mean()
df['Harga_Std_7']     = df['Harga'].rolling(window=7).std()
df['Harga_Lag_30']    = df['Harga'].shift(30)
df['RR_Lag_44']       = df['RR'].shift(44)
df = df.dropna().reset_index(drop=True)

print(f"Histori harga loaded: {len(df)} baris")
print(f"Data terakhir: {df['Tanggal'].max().date()}")

# ── 3. Ambil data cuaca dari Open-Meteo ─────────────────────
print("Mengambil data cuaca dari Open-Meteo...")

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude"  : 1.4748,
    "longitude" : 124.8421,
    "daily"     : [
        "temperature_2m_mean",
        "relative_humidity_2m_mean",
        "precipitation_sum"
    ],
    "timezone"  : "Asia/Makassar",
    "past_days" : 50,
    "forecast_days": 7
}

response = requests.get(url, params=params)
cuaca    = response.json()

df_cuaca = pd.DataFrame({
    'Tanggal' : pd.to_datetime(cuaca['daily']['time']),
    'TAVG'    : cuaca['daily']['temperature_2m_mean'],
    'RH_AVG'  : cuaca['daily']['relative_humidity_2m_mean'],
    'RR'      : cuaca['daily']['precipitation_sum']
})

today          = pd.Timestamp(datetime.now().date())
cuaca_hist     = df_cuaca[df_cuaca['Tanggal'] <  today].copy()
cuaca_forecast = df_cuaca[df_cuaca['Tanggal'] >= today].copy()

print(f"Data cuaca forecast: {len(cuaca_forecast)} hari ke depan")

# ── 4. Generate prediksi 7 hari ke depan ────────────────────
print("Generating prediksi...")

hasil_prediksi = []

harga_series = df['Harga'].tolist()
rr_series    = df_cuaca['RR'].tolist()

for i, row in cuaca_forecast.iterrows():
    tanggal = row['Tanggal']

    tavg   = row['TAVG']
    rh_avg = row['RH_AVG']
    rr     = row['RR']

    idx_lag44 = cuaca_hist[
        cuaca_hist['Tanggal'] == tanggal - timedelta(days=44)
    ]['RR']
    rr_lag44 = idx_lag44.values[0] if len(idx_lag44) > 0 else df['RR'].mean()

    harga_lag1  = harga_series[-1]
    harga_lag7  = harga_series[-7]  if len(harga_series) >= 7  else harga_series[0]
    harga_lag30 = harga_series[-30] if len(harga_series) >= 30 else harga_series[0]

    rolling7  = np.mean(harga_series[-7:])
    rolling14 = np.mean(harga_series[-14:]) if len(harga_series) >= 14 else np.mean(harga_series)
    std7      = np.std(harga_series[-7:])   if len(harga_series) >= 7  else np.std(harga_series)

    bulan        = tanggal.month
    tahun        = tanggal.year
    is_hari_raya = 1 if bulan in [12, 1] else 0

    input_data = pd.DataFrame([[
        rr, tavg, rh_avg, rr_lag44,
        harga_lag1, harga_lag7, harga_lag30,
        rolling7, rolling14, std7,
        bulan, tahun, is_hari_raya
    ]], columns=feature_cols)

    input_scaled = pd.DataFrame(
        scaler.transform(input_data),
        columns=feature_cols
    )

    pred  = model.predict(input_scaled)[0]
    margin         = pred * 0.08
    pred_low       = pred - margin
    pred_high      = pred + margin

    hasil_prediksi.append({
        'tanggal'            : tanggal.date(),
        'harga_prediksi'     : round(pred * 1000),
        'harga_prediksi_low' : round(pred_low * 1000),
        'harga_prediksi_high': round(pred_high * 1000)
    })

    harga_series.append(pred)

# ── 5. Simpan ke CSV ────────────────────────────────────────
df_hasil = pd.DataFrame(hasil_prediksi)
df_hasil.to_csv('prediksi_output.csv', index=False)

print("\n" + "="*50)
print("HASIL PREDIKSI 7 HARI KE DEPAN")
print("="*50)
print(df_hasil.to_string(index=False))
print("="*50)
print("Tersimpan di: scripts/prediksi_output.csv")

# ── 6. INSERT ke MySQL ───────────────────────────────────────
conn = mysql.connector.connect(
    host     = "localhost",
    user     = "root",
    password = "",
    database = "ricaholic"
)
cursor = conn.cursor()

for row in hasil_prediksi:
    cursor.execute("""
        INSERT INTO prediksi
        (tanggal, harga_prediksi, harga_bawah, harga_atas, created_at)
        VALUES (%s, %s, %s, %s, NOW())
        ON DUPLICATE KEY UPDATE
            harga_prediksi = VALUES(harga_prediksi),
            harga_bawah    = VALUES(harga_bawah),
            harga_atas     = VALUES(harga_atas)
    """, (
        row['tanggal'],
        row['harga_prediksi'],
        row['harga_prediksi_low'],
        row['harga_prediksi_high']
    ))

conn.commit()
cursor.close()
conn.close()
print("Prediksi berhasil disimpan ke MySQL.")

Model loaded.
Histori harga loaded: 2230 baris
Data terakhir: 2026-03-31
Mengambil data cuaca dari Open-Meteo...
Data cuaca forecast: 7 hari ke depan
Generating prediksi...

HASIL PREDIKSI 7 HARI KE DEPAN
   tanggal  harga_prediksi  harga_prediksi_low  harga_prediksi_high
2026-06-03           95984               88305               103662
2026-06-04           89405               82252                96557
2026-06-05           88158               81106                95211
2026-06-06           87916               80883                94949
2026-06-07           90121               82911                97330
2026-06-08           93203               85747               100659
2026-06-09           95243               87623               102862
Tersimpan di: scripts/prediksi_output.csv


InterfaceError: 2003: Can't connect to MySQL server on 'localhost:3306' (Errno 10061: No connection could be made because the target machine actively refused it)

In [ ]:
# ── 6. INSERT ke MySQL (aktifkan setelah DB siap) ───────────
# import mysql.connector
#
# conn = mysql.connector.connect(
#     host     = "localhost",
#     user     = "root",
#     password = "password_db",
#     database = "rica_manado"
# )
# cursor = conn.cursor()
#
# for row in hasil_prediksi:
#     cursor.execute("""
#         INSERT INTO prediksi 
#         (tanggal, harga_prediksi, harga_prediksi_low, 
#          harga_prediksi_high, created_at)
#         VALUES (%s, %s, %s, %s, NOW())
#         ON DUPLICATE KEY UPDATE
#             harga_prediksi      = VALUES(harga_prediksi),
#             harga_prediksi_low  = VALUES(harga_prediksi_low),
#             harga_prediksi_high = VALUES(harga_prediksi_high)
#     """, (
#         row['tanggal'],
#         row['harga_prediksi'],
#         row['harga_prediksi_low'],
#         row['harga_prediksi_high']
#     ))
#
# conn.commit()
# cursor.close()
# conn.close()
# print("Prediksi berhasil disimpan ke MySQL.")